# Predictive Waste Classification — Random Forest (Improved)
### Thread Dyeing Factory | Circular Supply Chain

**Improvements applied over baseline:**
| # | Technique | Detail |
|---|---|---|
| 1 | Lag & Rolling Features | lag-1, lag-7, roll-7, roll-30, std-7 for all 4 targets + production |
| 2 | Interaction Features | Prod×DyeBatch, Prod×Acid, Fixation_loss_kg, etc. |
| 3 | Skew Check | All 4 targets verified within ±0.5 → **no log transform needed** |
| 4 | Walk-Forward CV | TimeSeriesSplit(n_splits=5, gap=7) replaces random KFold |

**Workflow:** Train on 2023 → Test on 2024 | Targets: Solid_Waste_kg, Yarn_Waste_kg, Chemical_Waste_kg, Wastewater_L


In [ ]:
# ============================================================
# CELL 2: Install & Import Libraries
# ============================================================

# !pip install pandas numpy scikit-learn matplotlib seaborn openpyxl joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.inspection import permutation_importance
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
COLORS = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

OUTPUT_DIR = "output_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGETS = ['Solid_Waste_kg', 'Yarn_Waste_kg', 'Chemical_Waste_kg', 'Wastewater_L']

# Skew analysis on the updated dataset — ALL targets are within ±0.5,
# so NO log transform is applied (transforming near-normal data hurts performance):
#   Chemical_Waste_kg : skew = 0.43  → within ±0.5 → no transform
#   Solid_Waste_kg    : skew = 0.44  → within ±0.5 → no transform
#   Yarn_Waste_kg     : skew = 0.39  → within ±0.5 → no transform
#   Wastewater_L      : skew = 0.21  → within ±0.5 → no transform
LOG_TARGETS = set()   # empty → no target is log-transformed

print("✅ Libraries loaded successfully.")
print(f"📁 All figures will be saved to: '{OUTPUT_DIR}/'")


In [ ]:
# ============================================================
# CELL 3: Load Dataset
# ============================================================

FILE_PATH = "Textile_Dyeing_Dataset_Causal.xlsx"
SHEET     = "Sheet1"

df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

# Rename column with special character (causes issues in some sklearn ops)
df_raw = df_raw.rename(columns={'Liquor_Ratio_1:x': 'Liquor_Ratio_1x'})

print(f"✅ Dataset loaded and Dates converted | Shape: {df_raw.shape}")


In [ ]:
# ============================================================
# CELL 4: Data Exploration — Overview
# ============================================================

print("=" * 60)
print("BASIC DATASET INFO")
print("=" * 60)
df_raw.info()

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS (numeric columns)")
print("=" * 60)
display(df_raw.describe().T)

print("\n" + "=" * 60)
print("TARGET VARIABLE SKEWNESS — justifies log transform decision")
print("=" * 60)
for t in TARGETS:
    skew = df_raw[t].skew()
    flag = "→ LOG TRANSFORM applied" if t in LOG_TARGETS else "→ no transform needed"
    print(f"  {t:<25}: skew = {skew:.4f}  {flag}")


In [ ]:
# ============================================================
# CELL 5: Data Exploration — Missing Values
# ============================================================

missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values found in the dataset!")
else:
    print("⚠️  Missing values detected:")
    print(missing_df)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(df_raw.isnull(), yticklabels=False, cbar=True, cmap='viridis', ax=ax)
    ax.set_title("Missing Value Heatmap", fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/01_missing_value_heatmap.png", dpi=150)
    plt.show()
    print(f"📁 Saved: 01_missing_value_heatmap.png")


In [ ]:
# ============================================================
# CELL 6: Data Exploration — Distribution of Target Variables
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    axes[0, i].hist(df_raw[target].dropna(), bins=30, color=color, edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{target}\nHistogram', fontsize=11, fontweight='bold')
    axes[0, i].set_xlabel('Value')
    axes[0, i].set_ylabel('Frequency')

    axes[1, i].boxplot(df_raw[target].dropna(), patch_artist=True,
                       boxprops=dict(facecolor=color, alpha=0.7))
    axes[1, i].set_title(f'{target}\nBoxplot', fontsize=11, fontweight='bold')
    axes[1, i].set_ylabel('Value')

plt.suptitle("Distribution of Waste Target Variables", fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/02_target_distributions.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📁 Saved: 02_target_distributions.png")


In [ ]:
# ============================================================
# CELL 7: Data Exploration — Target Variables Over Time
# ============================================================

from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

df_sorted  = df_raw.sort_values('Date').copy()
split_date = pd.Timestamp('2024-01-01')

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    axes[i].plot(df_sorted['Date'], df_sorted[target], color=color, linewidth=0.8, alpha=0.8)
    axes[i].axvline(split_date, color='red', linestyle='--', linewidth=1.5, label='Train/Test Split')
    axes[i].set_ylabel(target, fontsize=10, fontweight='bold')
    if i == 0:
        axes[i].legend(fontsize=10)

axes[0].set_title("Waste Targets Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/03_targets_over_time.png", dpi=150)
plt.show()
print(f"📁 Saved: 03_targets_over_time.png")


In [ ]:
# ============================================================
# CELL 8: Data Exploration — Correlation Heatmap
# ============================================================

numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
corr_matrix  = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.3,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title("Full Feature Correlation Heatmap", fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/04_full_correlation_heatmap.png", dpi=150)
plt.show()
print(f"📁 Saved: 04_full_correlation_heatmap.png")

# Correlation of inputs with each target
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for i, target in enumerate(TARGETS):
    row, col = divmod(i, 2)
    corr_with_target = corr_matrix[target].drop(TARGETS).sort_values(key=abs, ascending=False)
    colors_bar = ['#D32F2F' if v > 0 else '#1565C0' for v in corr_with_target.values]
    axes[row, col].barh(corr_with_target.index, corr_with_target.values, color=colors_bar)
    axes[row, col].axvline(0, color='black', linewidth=0.8)
    axes[row, col].set_title(f'Top Correlations with {target}', fontsize=12, fontweight='bold')
    axes[row, col].set_xlabel('Pearson Correlation')

plt.suptitle("Input-Target Correlations (Top 20 Features per Target)", fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/05_input_target_correlations.png", dpi=150)
plt.show()
print(f"📁 Saved: 05_input_target_correlations.png")


## 🆕 IMPROVEMENT 1 & 2 — Feature Engineering
### What was added and why

| Group | Features | Reason |
|---|---|---|
| **Cyclical encoding** | Month_Sin/Cos, Day_Sin/Cos, Quarter | Month 1 and 12 are adjacent — raw numbers make them far apart. Sin/cos wraps the cycle correctly. |
| **Interaction features** | Prod×DyeBatch, Prod×Acid, Prod×Salt, Prod×Alkali, Prod×Auxiliary, Chem_inputs_total | High production *and* many batches together cause disproportionately more waste than either alone. |
| **Physically motivated** | Fixation_loss_kg = (1 − Dye_Fixation%) × Dye_Used_kg | Unfixed dye does not go onto fabric — it becomes chemical waste. This directly models the mechanism. |
| **Ratio features** | Chem_per_litre, Energy_per_worker, Prod_per_worker | Normalised efficiency features capture operational intensity. |
| **Lag features** | _lag1, _lag7 for all 4 targets + Production | Yesterday's waste tells the model about shift patterns and batch queues not visible in raw process variables. |
| **Rolling features** | _roll7, _roll30, _std7 | 7-day and 30-day context + volatility. All use `.shift(1)` to **prevent look-ahead data leakage**. |


In [ ]:
# ============================================================
# CELL 9: IMPROVEMENT 1 & 2 — Feature Engineering
#         Cyclical Encoding + Interaction Features + Lag/Rolling
# ============================================================

df = df_raw.sort_values('Date').copy().reset_index(drop=True)

# ── Cyclical time encoding ───────────────────────────────────
months      = df['Date'].dt.month
day_of_week = df['Date'].dt.dayofweek

df['Month_Sin'] = np.sin(2 * np.pi * months / 12)
df['Month_Cos'] = np.cos(2 * np.pi * months / 12)
df['Day_Sin']   = np.sin(2 * np.pi * day_of_week / 7)
df['Day_Cos']   = np.cos(2 * np.pi * day_of_week / 7)
df['Quarter']   = df['Date'].dt.quarter

# ── Interaction features ─────────────────────────────────────
df['Prod_x_DyeBatch']   = df['Production_Volume_kg'] * df['Dye_Batch_Count']
df['Prod_x_Acid']       = df['Production_Volume_kg'] * df['Acid_kg']
df['Prod_x_Salt']       = df['Production_Volume_kg'] * df['Salt_kg']
df['Prod_x_Alkali']     = df['Production_Volume_kg'] * df['Alkali_kg']
df['Prod_x_Auxiliary']  = df['Production_Volume_kg'] * df['Auxiliary_kg']
df['Chem_inputs_total'] = (df['Acid_kg'] + df['Salt_kg'] + df['Alkali_kg']
                           + df['Auxiliary_kg'] + df['Dye_Used_kg'])

# Physically motivated: unfixed dye → chemical waste
df['Fixation_loss_kg']  = (1 - df['Dye_Fixation_Ratio_%'] / 100) * df['Dye_Used_kg']
df['Chem_per_litre']    = df['Chem_inputs_total'] / (df['Total_Water_Consumption_L'] + 1)
df['Energy_per_worker'] = df['Total_Energy_kWh'] / (df['Number_of_Workers'] + 1)
df['Prod_per_worker']   = df['Production_Volume_kg'] / (df['Number_of_Workers'] + 1)

# ── Lag & rolling features ──
# FIX: build lag/rolling features SEPARATELY within each year so that no
# 2024 row ever uses 2023 values (leak-proof). .shift(1) prevents same-day leak.
LAG_COLS = [
    ('Chemical_Waste_kg',    'chem'),
    ('Solid_Waste_kg',       'solid'),
    ('Yarn_Waste_kg',        'yarn'),
    ('Wastewater_L',         'water'),
    ('Production_Volume_kg', 'prod'),
]

def _add_lag_features(g):
    g = g.sort_values('Date').copy()
    for col, prefix in LAG_COLS:
        s = g[col].shift(1)
        g[f'{prefix}_lag1']   = s
        g[f'{prefix}_lag7']   = g[col].shift(7)
        g[f'{prefix}_roll7']  = s.rolling(7,  min_periods=1).mean()
        g[f'{prefix}_roll30'] = s.rolling(30, min_periods=1).mean()
        g[f'{prefix}_std7']   = s.rolling(7,  min_periods=1).std().fillna(0)
    return g

# Compute within each calendar year independently -> train (2023) and test (2024)
# never share lag information across the split boundary.
df = (df.groupby(df['Date'].dt.year, group_keys=False)
        .apply(_add_lag_features)
        .reset_index(drop=True))

# Drop rows with NaN from lag features (first ~30 rows of EACH year)
df = df.dropna().reset_index(drop=True)

# ── Final feature list (dynamic — auto-includes all new features) ──
EXCL = (TARGETS
        + ['Solid_Waste_%_of_Production', 'Yarn_Waste_%_of_Production',
           'Chemical_Waste_kg_per_1000kg', 'Wastewater_L_per_kg',
           'Date', 'Day_Type', 'Year', 'Month'])

FEATURE_COLS = [c for c in df.columns if c not in EXCL]

print(f"✅ Feature engineering complete.")
print(f"   Original features : 23")
print(f"   New features added: {len(FEATURE_COLS) - 23}")
print(f"   Total features    : {len(FEATURE_COLS)}")
print(f"\nNew features added:")
original_set = {'Month_Sin','Month_Cos','Day_Sin','Day_Cos','Quarter',
                'Production_Volume_kg','Machine_Utilization_%','Number_of_Workers',
                'Total_Energy_kWh','Total_Water_Consumption_L','Dye_Fixation_Ratio_%',
                'Liquor_Ratio_1x','Temperature_C','Cycle_Time_hr','Dye_Batch_Count',
                'Dye_Used_kg','Salt_kg','Alkali_kg','Acid_kg','Auxiliary_kg',
                'BOD_mg_per_L','COD_mg_per_L','pH'}
for f in FEATURE_COLS:
    if f not in original_set:
        print(f"  + {f}")


In [ ]:
# ============================================================
# CELL 10: Train / Test Split (Year-Based)
# ============================================================

df_train = df[df['Date'].dt.year == 2023].copy()
df_test  = df[df['Date'].dt.year == 2024].copy()

X_train = df_train[FEATURE_COLS].reset_index(drop=True)
y_train = df_train[TARGETS].reset_index(drop=True)
X_test  = df_test[FEATURE_COLS].reset_index(drop=True)
y_test  = df_test[TARGETS].reset_index(drop=True)

print(f"✅ Train set (2023): X={X_train.shape}, y={y_train.shape}")
print(f"✅ Test  set (2024): X={X_test.shape},  y={y_test.shape}")
print(f"\n   NaN in X_train : {X_train.isnull().sum().sum()}")
print(f"   NaN in X_test  : {X_test.isnull().sum().sum()}")


## 🆕 IMPROVEMENT 3 — Skew Check (No Log Transform Needed)

**Decision rule:** a skew between **−0.5 and +0.5** is considered approximately
normal for modelling. Applying a log transform to an already-normal distribution
*distorts* it and can hurt performance, so we only transform when skew exceeds ±0.5.

On the updated dataset, **all four targets fall inside ±0.5**, so no log transform
is applied to any target:

| Target | Skew | Decision |
|---|---|---|
| Chemical_Waste_kg | 0.43 | ❌ No transform — within ±0.5 |
| Solid_Waste_kg | 0.44 | ❌ No transform — within ±0.5 |
| Yarn_Waste_kg | 0.39 | ❌ No transform — within ±0.5 |
| Wastewater_L | 0.21 | ❌ No transform — within ±0.5 |

`LOG_TARGETS` is therefore an empty set, and every target is modelled and
reported on its **original scale**.


In [ ]:
# ============================================================
# CELL 11: IMPROVEMENT 3 — Skew Check (no log transform applied)
# ============================================================

# LOG_TARGETS is empty for this dataset, so y_train_transformed == y_train.
# Kept for compatibility with the downstream predict/inverse-transform code.
y_train_transformed = y_train.copy()
for t in LOG_TARGETS:                      # empty → loop does nothing
    y_train_transformed[t] = np.log1p(y_train[t])

print("Skew check — log transform decision per target:")
print("-" * 55)
for t in TARGETS:
    skew = y_train[t].skew()
    if t in LOG_TARGETS:
        print(f"  {t:<25}: {skew:.3f}  -> log transformed")
    else:
        print(f"  {t:<25}: {skew:.3f}  -> within +/-0.5, NOT transformed")


In [ ]:
# ============================================================
# CELL 12: Baseline Random Forest Model (Default Parameters)
# ============================================================

rf_base    = RandomForestRegressor(n_estimators=300, random_state=42,
                                   n_jobs=-1, oob_score=True)
model_base = MultiOutputRegressor(rf_base, n_jobs=-1)
model_base.fit(X_train, y_train_transformed)

# Predict and reverse log transform where applied
y_pred_base_raw = pd.DataFrame(model_base.predict(X_test), columns=TARGETS)
y_pred_base = y_pred_base_raw.copy()
for t in LOG_TARGETS:
    y_pred_base[t] = np.expm1(y_pred_base_raw[t])

print("✅ Baseline model trained.")
print("\n─── Baseline Test Performance ───────────────────────────────")
for target in TARGETS:
    mae  = mean_absolute_error(y_test[target], y_pred_base[target])
    rmse = np.sqrt(mean_squared_error(y_test[target], y_pred_base[target]))
    r2   = r2_score(y_test[target], y_pred_base[target])
    mape = mean_absolute_percentage_error(y_test[target], y_pred_base[target]) * 100
    print(f"  {target:<25}  MAE={mae:>10.2f}  RMSE={rmse:>10.2f}  R²={r2:.4f}  MAPE={mape:.2f}%")


## 🆕 IMPROVEMENT 4 — Walk-Forward Time-Series Cross-Validation

**Why NOT regular KFold?**

Your data is daily time-series (2023–2024). Regular `KFold(shuffle=True)` randomly mixes past and future rows — this means the model can accidentally train on future data to predict past data. That is **data leakage** and produces artificially optimistic R² scores.

**Walk-Forward CV** always trains on the past and tests on the future — exactly how a real deployed model would operate.

```
Fold 1: Train [Jan–Apr 2023]  →  Test [Jun–Jul 2023]   (gap=7 days)
Fold 2: Train [Jan–Jul 2023]  →  Test [Sep–Oct 2023]
Fold 3: Train [Jan–Oct 2023]  →  Test [Dec 2023]
...
```

The `gap=7` is important — it prevents the 7-day lag features from bleeding across the fold boundary.

---

**Per-target tuning (Option B):** rather than tuning on a single target and reusing those parameters for all four, the grid search is run **separately for each target**. Each waste stream then gets the hyperparameters that generalise best for *it* under walk-forward CV. This avoids the situation where one target's optimal settings hurt another (e.g. settings best for solid waste underperforming on wastewater).


In [ ]:
# ============================================================
# CELL 13: IMPROVEMENT 4 — Walk-Forward CV + Per-Target Hyperparameter Tuning
# NOTE: This may take 15-40 minutes (4 separate grid searches).
#
# REVISED TUNING PROTOCOL (identical structure to the XGBoost notebook):
#   - n_splits=3 (was 5): with only ~358 training rows, 5 splits produced
#     tiny early folds whose scores did not transfer to the full-data fit.
#   - scoring='neg_mean_absolute_error' (was 'r2'): MAE is a far more stable
#     selection signal than R2 on ~60-row validation folds.
#   - Grid INCLUDES the strong RF defaults (max_depth=None, min_samples_leaf=1,
#     max_features=1.0) so the search can select them if they are best. RF
#     relies on deep, fully-grown trees decorrelated by bagging; constraining
#     depth/leaf size hurts RF (unlike XGBoost). This keeps the protocol
#     symmetric while letting each algorithm use its own optimal configuration.
# ============================================================

tscv = TimeSeriesSplit(n_splits=3, gap=7)   # gap=7 prevents lag feature leakage

param_grid = {
    'n_estimators'     : [400, 600],
    'max_depth'        : [None, 20],            # None RESTORED - RF relies on deep trees
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2],                # 1 RESTORED - RF's strong default
    'max_features'     : ['sqrt', 0.5, 1.0],    # 1.0 included - default full-feature path
}

print("\u23f3 Running Walk-Forward GridSearchCV SEPARATELY for each target...")
print("   Each target gets its own best hyperparameters (Option B).")
print("   CV = TimeSeriesSplit(n_splits=3, gap=7); scoring = MAE (stable on small folds).\n")

best_params_per_target = {}
cv_score_per_target    = {}

for target in TARGETS:
    print(f"   -> Tuning: {target} ...")
    gs = GridSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=-1),
        param_grid,
        scoring='neg_mean_absolute_error',   # MAE: matches XGBoost protocol, less noisy than r2
        cv=tscv,          # walk-forward, NOT random KFold
        n_jobs=-1,
        verbose=0,
        refit=True
    )
    gs.fit(X_train, y_train_transformed[target])
    best_params_per_target[target] = gs.best_params_
    cv_score_per_target[target]    = gs.best_score_   # this is -MAE (higher = better)
    print(f"      best CV score (neg-MAE) = {gs.best_score_:.4f}")
    print(f"      best params = {gs.best_params_}\n")

# Keep a single 'best_params' for any legacy reference (uses first target).
best_params = best_params_per_target[TARGETS[0]]

print("\u2705 Per-target tuning complete. Summary:")
print("-" * 60)
for target in TARGETS:
    print(f"  {target:<25}: CV neg-MAE={cv_score_per_target[target]:.4f}")

In [ ]:
# ============================================================
# CELL 14: Final Tuned Model - Per-Target Training & Prediction
# ============================================================

class PerTargetRegressor:
    """Trains one RandomForest per target, each with its OWN best params.
    Exposes .estimators_ (in TARGETS order) so it is a drop-in replacement
    for MultiOutputRegressor in the downstream cells."""
    def __init__(self, params_per_target, targets):
        self.params_per_target = params_per_target
        self.targets = targets
        self.estimators_ = []

    def fit(self, X, Y):
        self.estimators_ = []
        for t in self.targets:
            est = RandomForestRegressor(**self.params_per_target[t],
                                        random_state=42, n_jobs=-1, oob_score=True)
            est.fit(X, Y[t])
            self.estimators_.append(est)
        return self

    def predict(self, X):
        preds = [est.predict(X) for est in self.estimators_]
        return np.column_stack(preds)

model_final = PerTargetRegressor(best_params_per_target, TARGETS)
model_final.fit(X_train, y_train_transformed)

# Predict - reverse log transform only if any LOG_TARGETS (empty here)
y_pred_raw = pd.DataFrame(model_final.predict(X_test), columns=TARGETS)
y_pred_df  = y_pred_raw.copy()
for t in LOG_TARGETS:
    y_pred_df[t] = np.expm1(y_pred_raw[t])

print("\u2705 Per-target tuned model trained and predictions made on 2024 test data.")
print("   Each target used its own optimised hyperparameters.")


In [ ]:
# ============================================================
# CELL 15: Model Evaluation — Full Metrics Table
# ============================================================

metrics_rows = []
print("=" * 80)
print(f"{'Target':<30} {'MAE':>9} {'RMSE':>10} {'R²':>8} {'MAPE%':>8}")
print("=" * 80)

for target in TARGETS:
    mae  = mean_absolute_error(y_test[target], y_pred_df[target])
    mse  = mean_squared_error(y_test[target], y_pred_df[target])
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test[target], y_pred_df[target])
    mape = mean_absolute_percentage_error(y_test[target], y_pred_df[target]) * 100
    metrics_rows.append({'Target': target, 'MAE': round(mae, 3),
                         'MSE': round(mse, 3), 'RMSE': round(rmse, 3),
                         'R2': round(r2, 4), 'MAPE_%': round(mape, 2)})
    print(f"  {target:<28} {mae:>10.3f} {rmse:>10.3f} {r2:>8.4f} {mape:>7.2f}%")
print("=" * 80)

metrics_df = pd.DataFrame(metrics_rows)
print("\n📊 Full Metrics DataFrame:")
display(metrics_df)
metrics_df.to_csv(f"{OUTPUT_DIR}/model_metrics.csv", index=False)
print(f"\n📁 Metrics saved to: {OUTPUT_DIR}/model_metrics.csv")


In [ ]:
# ============================================================
# CELL 16: Metrics Comparison — Baseline vs Tuned
# ============================================================

metrics_base_rows, metrics_tuned_rows = [], []
for target in TARGETS:
    metrics_base_rows.append({
        'Target': target,
        'MAE' : mean_absolute_error(y_test[target], y_pred_base[target]),
        'RMSE': np.sqrt(mean_squared_error(y_test[target], y_pred_base[target])),
        'R2'  : r2_score(y_test[target], y_pred_base[target]),
        'Model': 'Baseline RF'
    })
    metrics_tuned_rows.append({
        'Target': target,
        'MAE' : mean_absolute_error(y_test[target], y_pred_df[target]),
        'RMSE': np.sqrt(mean_squared_error(y_test[target], y_pred_df[target])),
        'R2'  : r2_score(y_test[target], y_pred_df[target]),
        'Model': 'Tuned RF (Improved)'
    })

compare_df = pd.DataFrame(metrics_base_rows + metrics_tuned_rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
    pivot = compare_df.pivot(index='Target', columns='Model', values=metric)
    pivot.plot(kind='bar', ax=ax, color=['#90CAF9', '#1565C0'], edgecolor='white')
    ax.set_title(f'{metric} Comparison', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=25)
    ax.legend(fontsize=9)

plt.suptitle("Baseline vs Tuned Random Forest Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/06_baseline_vs_tuned.png", dpi=150)
plt.show()
print(f"📁 Saved: 06_baseline_vs_tuned.png")


In [ ]:
# ============================================================
# CELL 17: Visual — Actual vs Predicted Scatter (All 4 Targets)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    row, col = divmod(i, 2)
    ax = axes[row, col]

    y_true_col = y_test[target].values
    y_pred_col = y_pred_df[target].values

    ax.scatter(y_true_col, y_pred_col, alpha=0.5, color=color, edgecolors='white', s=20)
    min_val = min(y_true_col.min(), y_pred_col.min())
    max_val = max(y_true_col.max(), y_pred_col.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect Fit')

    r2  = r2_score(y_true_col, y_pred_col)
    mae = mean_absolute_error(y_true_col, y_pred_col)
    ax.set_title(f'{target}\nR²={r2:.4f}   MAE={mae:.2f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.legend(fontsize=9)

plt.suptitle("Actual vs Predicted — Tuned Random Forest (Test 2024)", fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/07_actual_vs_predicted_scatter.png", dpi=150)
plt.show()
print(f"📁 Saved: 07_actual_vs_predicted_scatter.png")


In [ ]:
# ============================================================
# CELL 18: Visual — Predicted vs Actual Time Series (2024)
# ============================================================

date_test = df_test['Date'].reset_index(drop=True)

fig, axes = plt.subplots(4, 1, figsize=(18, 16), sharex=True)
for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    axes[i].plot(date_test, y_test[target].values,    color='grey', linewidth=1,
                 label='Actual', alpha=0.7)
    axes[i].plot(date_test, y_pred_df[target].values, color=color,  linewidth=1,
                 label='Predicted', alpha=0.9, linestyle='--')
    axes[i].set_ylabel(target, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=9, loc='upper right')
    r2 = r2_score(y_test[target], y_pred_df[target])
    axes[i].set_title(f'{target}  (R²={r2:.4f})', fontsize=10)

plt.suptitle("Time Series: Actual vs Predicted Waste (2024 Test Period)",
             fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/08_timeseries_actual_vs_predicted.png", dpi=150)
plt.show()
print(f"📁 Saved: 08_timeseries_actual_vs_predicted.png")


In [ ]:
# ============================================================
# CELL 19: Residual Analysis
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    residuals = y_test[target].values - y_pred_df[target].values

    axes[0, i].scatter(y_pred_df[target].values, residuals, alpha=0.4, color=color)
    axes[0, i].axhline(0, color='black', linewidth=1, linestyle='--')
    axes[0, i].set_title(f'{target}\nResiduals vs Predicted', fontsize=9, fontweight='bold')
    axes[0, i].set_xlabel('Predicted')
    axes[0, i].set_ylabel('Residual')

    axes[1, i].hist(residuals, bins=30, color=color, edgecolor='white', alpha=0.8)
    axes[1, i].axvline(0, color='black', linewidth=1.5, linestyle='--')
    axes[1, i].set_title(f'{target}\nResidual Distribution', fontsize=9, fontweight='bold')
    axes[1, i].set_xlabel('Residual')
    axes[1, i].set_ylabel('Frequency')

plt.suptitle("Residual Analysis — Tuned Random Forest", fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/09_residual_analysis.png", dpi=150)
plt.show()
print(f"📁 Saved: 09_residual_analysis.png")


In [ ]:
# ============================================================
# CELL 20: Feature Importance — Per Target
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(20, 16))
feature_importance_dict = {}

for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    estimator   = model_final.estimators_[i]
    importances = estimator.feature_importances_
    indices     = np.argsort(importances)[::-1]
    feature_importance_dict[target] = pd.Series(importances, index=FEATURE_COLS)

    row, col  = divmod(i, 2)
    ax        = axes[row, col]
    top_n     = 15
    top_feats = [FEATURE_COLS[j] for j in indices[:top_n]]
    top_imp   = importances[indices[:top_n]]

    ax.barh(range(top_n), top_imp[::-1], color=color, edgecolor='white', alpha=0.85)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_feats[::-1], fontsize=9)
    ax.set_xlabel('Importance Score', fontsize=10)
    ax.set_title(f'Feature Importance: {target}', fontsize=11, fontweight='bold')

plt.suptitle("Top 15 Feature Importances per Waste Target", fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/10_feature_importance_per_target.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📁 Saved: 10_feature_importance_per_target.png")


In [ ]:
# ============================================================
# CELL 21: Aggregated Feature Importance (Across All Targets)
# ============================================================

all_importances = pd.DataFrame(feature_importance_dict)
mean_importance = all_importances.mean(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(mean_importance.index[::-1], mean_importance.values[::-1],
        color='#1565C0', edgecolor='white', alpha=0.85)
ax.set_xlabel('Mean Importance Score (Across All 4 Targets)', fontsize=11)
ax.set_title('Aggregated Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/11_aggregated_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📁 Saved: 11_aggregated_feature_importance.png")

print("\n📊 Mean Feature Importance Ranking:")
for rank, (feat, imp) in enumerate(mean_importance.items(), 1):
    print(f"    {rank:2d}. {feat:<35} {imp:.4f}")


In [ ]:
# ============================================================
# CELL 22: Permutation Importance (More Reliable than Built-in)
# ============================================================

print("⏳ Computing permutation importance (may take ~1-2 minutes)...")

perm_imp_dict = {}
for i, target in enumerate(TARGETS):
    estimator = model_final.estimators_[i]
    y_eval    = (np.log1p(y_test[target].values)
                 if target in LOG_TARGETS else y_test[target].values)
    result    = permutation_importance(estimator, X_test, y_eval,
                                       n_repeats=10, random_state=42, n_jobs=-1)
    perm_imp_dict[target] = pd.Series(result.importances_mean, index=FEATURE_COLS)
    print(f"   ✅ Done: {target}")

perm_df   = pd.DataFrame(perm_imp_dict)
perm_mean = perm_df.mean(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(perm_mean.index[::-1], perm_mean.values[::-1],
        color='#388E3C', edgecolor='white', alpha=0.85)
ax.set_xlabel('Mean Permutation Importance (Across All 4 Targets)', fontsize=11)
ax.set_title('Permutation Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/12_permutation_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📁 Saved: 12_permutation_importance.png")


In [ ]:
# ============================================================
# CELL 23: Walk-Forward CV - Final Stability Report
# (Report these R^2 values in your paper's methodology section)
# Each target evaluated with ITS OWN best params (Option B).
# ============================================================

print("\u23f3 Running Walk-Forward CV on all 4 targets with per-target best params...")

cv_results = {}
for target in TARGETS:
    scores = []
    y_cv   = y_train_transformed[target].values
    params = best_params_per_target[target]

    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
        X_tr  = X_train.iloc[tr_idx]
        X_val = X_train.iloc[val_idx]
        y_tr  = y_cv[tr_idx]
        y_val_orig = y_train[target].values[val_idx]

        est = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        est.fit(X_tr, y_tr)
        y_pred_val = est.predict(X_val)
        if target in LOG_TARGETS:
            y_pred_val = np.expm1(y_pred_val)
        scores.append(r2_score(y_val_orig, y_pred_val))

    cv_results[target] = scores
    print(f"   {target:<25}  CV R2: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

cv_df = pd.DataFrame(cv_results)

fig, ax = plt.subplots(figsize=(12, 5))
cv_df.boxplot(ax=ax, patch_artist=True,
              boxprops=dict(facecolor='#90CAF9'),
              medianprops=dict(color='#0D47A1', linewidth=2))
ax.axhline(0.8, color='orange', linestyle='--', linewidth=1, label='R2=0.8 threshold')
ax.axhline(0.5, color='red',    linestyle='--', linewidth=1, label='R2=0.5 threshold')
ax.set_title("5-Fold Walk-Forward Cross-Validation R2 (Training Data 2023)",
             fontsize=12, fontweight='bold')
ax.set_ylabel("R2 Score")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/13_walk_forward_cv_r2.png", dpi=150)
plt.show()
print(f"\U0001F4C1 Saved: 13_walk_forward_cv_r2.png")


In [ ]:
# ============================================================
# CELL 24: Monthly Performance Analysis
# ============================================================

month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

df_eval = df_test[['Date','Month']].copy().reset_index(drop=True)
df_eval = pd.concat([df_eval,
                     y_test.reset_index(drop=True),
                     y_pred_df.reset_index(drop=True)], axis=1)
df_eval.columns = (list(df_test[['Date','Month']].columns)
                   + [f"{t}_Actual"    for t in TARGETS]
                   + [f"{t}_Predicted" for t in TARGETS])

monthly_r2 = df_eval.groupby('Month').apply(
    lambda g: pd.Series({
        t: r2_score(g[f'{t}_Actual'], g[f'{t}_Predicted']) for t in TARGETS
    })
).reset_index()
monthly_r2['Month_num'] = monthly_r2['Month'].apply(lambda x: month_order.index(x))
monthly_r2 = monthly_r2.sort_values('Month_num')

fig, ax = plt.subplots(figsize=(14, 6))
for target, color in zip(TARGETS, COLORS):
    ax.plot(monthly_r2['Month'], monthly_r2[target],
            marker='o', label=target, color=color, linewidth=2)
ax.axhline(0.9, color='gray', linestyle='--', linewidth=1)
ax.set_title("Monthly R² Score per Waste Target (Test Year 2024)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Month")
ax.set_ylabel("R² Score")
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/14_monthly_r2.png", dpi=150)
plt.show()
print(f"📁 Saved: 14_monthly_r2.png")


In [ ]:
# ============================================================
# CELL 25: Save Final Model & Predictions
# ============================================================

model_path = f"{OUTPUT_DIR}/random_forest_waste_model_improved.pkl"
joblib.dump(model_final, model_path)
print(f"\u2705 Model saved: {model_path}")

# Save the per-target best hyperparameters (portable, reload-safe)
params_path = f"{OUTPUT_DIR}/best_params_per_target.pkl"
joblib.dump(best_params_per_target, params_path)
print(f"\u2705 Per-target best params saved: {params_path}")

results_df = df_test[['Date','Month']].reset_index(drop=True).copy()
for target in TARGETS:
    results_df[f'{target}_Actual']    = y_test[target].values
    results_df[f'{target}_Predicted'] = y_pred_df[target].values
    results_df[f'{target}_Error']     = y_test[target].values - y_pred_df[target].values

results_path = f"{OUTPUT_DIR}/predictions_2024_improved.csv"
results_df.to_csv(results_path, index=False)
print(f"\u2705 Predictions saved: {results_path}")


In [ ]:
# ============================================================
# CELL 26: Automated Future Prediction Template
# ============================================================

def predict_new_data(date_string, factory_values):
    """
    date_string   : 'YYYY-MM-DD'
    factory_values: dict of raw process variables
    """
    input_df = pd.DataFrame([factory_values])
    input_df['Date'] = pd.to_datetime(date_string)

    m  = input_df['Date'].dt.month.values[0]
    dw = input_df['Date'].dt.dayofweek.values[0]
    input_df['Month_Sin']        = np.sin(2*np.pi*m/12)
    input_df['Month_Cos']        = np.cos(2*np.pi*m/12)
    input_df['Day_Sin']          = np.sin(2*np.pi*dw/7)
    input_df['Day_Cos']          = np.cos(2*np.pi*dw/7)
    input_df['Quarter']          = input_df['Date'].dt.quarter
    input_df['Prod_x_DyeBatch']  = input_df['Production_Volume_kg'] * input_df['Dye_Batch_Count']
    input_df['Prod_x_Acid']      = input_df['Production_Volume_kg'] * input_df['Acid_kg']
    input_df['Prod_x_Salt']      = input_df['Production_Volume_kg'] * input_df['Salt_kg']
    input_df['Prod_x_Alkali']    = input_df['Production_Volume_kg'] * input_df['Alkali_kg']
    input_df['Prod_x_Auxiliary'] = input_df['Production_Volume_kg'] * input_df['Auxiliary_kg']
    input_df['Chem_inputs_total']= (input_df['Acid_kg'] + input_df['Salt_kg']
                                    + input_df['Alkali_kg'] + input_df['Auxiliary_kg']
                                    + input_df['Dye_Used_kg'])
    input_df['Fixation_loss_kg'] = (1 - input_df['Dye_Fixation_Ratio_%']/100) * input_df['Dye_Used_kg']
    input_df['Chem_per_litre']   = input_df['Chem_inputs_total'] / (input_df['Total_Water_Consumption_L']+1)
    input_df['Energy_per_worker']= input_df['Total_Energy_kWh'] / (input_df['Number_of_Workers']+1)
    input_df['Prod_per_worker']  = input_df['Production_Volume_kg'] / (input_df['Number_of_Workers']+1)

    # Use last-30-day means as proxy for lag/rolling features
    for col, prefix in LAG_COLS:
        last_mean = df[col].tail(30).mean()
        last_std  = df[col].tail(30).std()
        input_df[f'{prefix}_lag1']   = last_mean
        input_df[f'{prefix}_lag7']   = last_mean
        input_df[f'{prefix}_roll7']  = last_mean
        input_df[f'{prefix}_roll30'] = last_mean
        input_df[f'{prefix}_std7']   = last_std

    preds = model_final.predict(input_df[FEATURE_COLS])[0]

    print(f"\n🔮 WASTE PREDICTION FOR: {date_string}")
    print("-" * 40)
    for target, val in zip(TARGETS, preds):
        actual_val = np.expm1(val) if target in LOG_TARGETS else val
        print(f"  🏭 {target:<25}: {actual_val:.2f}")


# ── Example usage ──────────────────────────────────────────
user_inputs = {
    'Production_Volume_kg': 15300, 'Machine_Utilization_%': 85,
    'Number_of_Workers': 100,      'Total_Energy_kWh': 25000,
    'Total_Water_Consumption_L': 750000, 'Dye_Fixation_Ratio_%': 65,
    'Liquor_Ratio_1x': 8.5,        'Temperature_C': 65,
    'Cycle_Time_hr': 6,            'Dye_Batch_Count': 280,
    'Dye_Used_kg': 50,             'Salt_kg': 320,
    'Alkali_kg': 110,              'Acid_kg': 30,
    'Auxiliary_kg': 80,            'BOD_mg_per_L': 200,
    'COD_mg_per_L': 850,           'pH': 8.5,
    'Energy_Consumption_kWh_per_kg': 1.67,
    'Water_Consumption_L_per_kg': 50,
    'Production_Volume_ton': 15,
    'Monthly_Demand_Factor': 1.0,
    'Dye_Used_g_per_kg': 27, 'Salt_g_per_kg': 280,
    'Alkali_g_per_kg': 73,   'Acid_g_per_kg': 50,
    'Auxiliary_g_per_kg': 70,
}
predict_new_data('2025-07-15', user_inputs)


In [ ]:
# ============================================================
# CELL 27: Summary Dashboard Figure
# ============================================================

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

ax_r2 = fig.add_subplot(gs[0, :2])
r2_scores = [r2_score(y_test[t], y_pred_df[t]) for t in TARGETS]
bars = ax_r2.bar(TARGETS, r2_scores, color=COLORS, edgecolor='white', alpha=0.9)
ax_r2.axhline(0.9, color='gray', linestyle='--', linewidth=1)
ax_r2.set_ylim(0, 1.05)
ax_r2.set_title("R² Scores per Target (Test 2024)", fontsize=11, fontweight='bold')
ax_r2.set_ylabel("R²")
ax_r2.tick_params(axis='x', rotation=20)
for bar, score in zip(bars, r2_scores):
    ax_r2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
               f'{score:.3f}', ha='center', va='bottom', fontsize=9)

ax_mape = fig.add_subplot(gs[0, 2:])
mape_scores = [mean_absolute_percentage_error(y_test[t], y_pred_df[t]) * 100 for t in TARGETS]
ax_mape.bar(TARGETS, mape_scores, color=COLORS, edgecolor='white', alpha=0.9)
ax_mape.set_title("MAPE% per Target (Test 2024)", fontsize=11, fontweight='bold')
ax_mape.set_ylabel("MAPE %")
ax_mape.tick_params(axis='x', rotation=20)

for i, (target, color) in enumerate(zip(TARGETS, COLORS)):
    row = 1 + i // 2
    col = (i % 2) * 2
    ax  = fig.add_subplot(gs[row, col:col+2])
    y_true_col = y_test[target].values
    y_pred_col = y_pred_df[target].values
    ax.scatter(y_true_col, y_pred_col, alpha=0.4, color=color, s=15)
    mn = min(y_true_col.min(), y_pred_col.min())
    mx = max(y_true_col.max(), y_pred_col.max())
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1)
    r2 = r2_score(y_true_col, y_pred_col)
    ax.set_title(f'{target} | R²={r2:.4f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Actual', fontsize=8)
    ax.set_ylabel('Predicted', fontsize=8)

plt.suptitle("Waste Prediction Model — Summary Dashboard (Random Forest Improved)",
             fontsize=14, fontweight='bold')
fig.savefig(f"{OUTPUT_DIR}/15_summary_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📁 Saved: 15_summary_dashboard.png")


In [ ]:
# ============================================================
# CELL 28: Reverse Logistics — Treated Wastewater CSV
# ============================================================

rl_df = results_df.copy()
rl_df['Treated_Wastewater_L'] = (
    rl_df['Wastewater_L_Predicted'] -
    (rl_df['Chemical_Waste_kg_Predicted'] * 0.8)
).clip(lower=0)

rl_path = f"{OUTPUT_DIR}/predictions_2024_RL_improved.csv"
rl_df.to_csv(rl_path, index=False)
print(f"✅ New file saved: {rl_path}")
print(f"   New column    : Treated_Wastewater_L")
print(f"   Total columns : {len(rl_df.columns)}")

print("\n" + "=" * 60)
print("🚀 ALL CELLS COMPLETED SUCCESSFULLY!")
print(f"   All outputs saved to: '{OUTPUT_DIR}/'")
print("=" * 60)
print("\nSUMMARY OF IMPROVEMENTS:")
print("  1. Lag & Rolling Features  → 25 new time-series features")
print("  2. Interaction Features    → Prod×DyeBatch, Fixation_loss_kg, etc.")
print("  3. Log Transform           → Chemical_Waste_kg ONLY (skew=0.59)")
print("  4. Walk-Forward CV         -> TimeSeriesSplit(n_splits=5, gap=7) + PER-TARGET tuning")
